[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/18.%20The%20Yard%20Crane%20%28RTG_RMG%29%20Scheduling%20Problem/P18-Tier-2_executed.ipynb)

---


# 18. The Yard Crane (RTG/RMG) Scheduling Problem
## Tier 2: The Cheapest Insertion Heuristic

### Problem, inputs, and outputs

A yard crane must perform container-handling jobs at specified bay positions. Each job has a processing time, a release time (the earliest time it may begin), a due date, and a priority weight. The input also specifies each crane's starting position. The output is an ordered route for each crane, together with estimated job start and completion times and summary performance measures.

The notebook constructs a schedule using the **cheapest insertion heuristic**. It starts with an empty route for every crane. At each iteration, it considers every unscheduled job, every crane, and every position in that crane's current route. It inserts the job-and-position combination with the smallest increase in the route objective, then repeats until all jobs have been assigned.

### Intuition and modeling assumptions

An insertion is a local change: a job is placed between two consecutive route nodes. The cost of that change depends on both the extra travel and the effect of the new route order on job completion times. Because moving a job can delay jobs already in the route, the candidate route is fully rescheduled before its cost is evaluated.

The model uses a one-dimensional bay line. Travel time between positions is the absolute difference in bay positions, at one minute per bay. A crane travels to a job, waits if it arrives before the job's release time, and then processes it. Jobs are assigned to exactly one crane. This example does not model interference between cranes, blocking, setup times, shift limits, uncertain travel or processing times, or changes in jobs after the schedule is built.

### Objective and insertion cost

For a route \(R\), the route objective combines travel time with priority-weighted tardiness:

$$\
F(R) = \sum_{(u,v)\in R} |p_u-p_v|
+ \sum_{j\in R} w_j\max(0,C_j-d_j).
$$

Here, \(p_u\) and \(p_v\) are consecutive node positions, including the start and end nodes; \(C_j\) is job \(j\)'s completion time; \(d_j\) is its due date; and \(w_j\) is its priority weight. Travel is measured in minutes, while weighted tardiness is measured in weighted minutes. The objective is a modeling score that adds these two components; it is not a monetary cost.

For a candidate insertion of job \(j\) into route \(R\), the heuristic evaluates

$$\
\Delta F = F(R\mathbin{\oplus}j)-F(R),
$$

where \(R\mathbin{\oplus}j\) denotes the route with \(j\) inserted at the candidate position and with all its job times recalculated. The algorithm chooses the candidate with the smallest \(\Delta F\). If two candidates have the same cost, the first encountered in the deterministic loop order is retained. This greedy rule does not guarantee a globally optimal schedule.

### Timing equations

For a job \(j\) following node \(i\), its arrival, start, and completion times are

$$\
A_j = C_i + |p_i-p_j|, \qquad
S_j = \max(A_j,r_j), \qquad
C_j = S_j + q_j.
$$

The start node has completion time zero. Here \(r_j\) is the release time and \(q_j\) is the processing time. The end node's arrival time is the preceding node's completion time plus travel back to the crane's starting position. These equations ensure that the schedule accounts for both travel and release-time waiting. In the code, `schedule_route` applies these equations, while `compute_insertion_cost` compares the objective before and after a candidate insertion.

### Algorithm and computational effort

The initial route for each crane contains only a start node and an end node at the crane's starting position. If there are \(n\) jobs and \(m\) cranes, the algorithm performs \(n\) insertion iterations. In an iteration with \(k\) jobs already scheduled, it examines up to \(m(k+1)\) insertions. Each candidate route is rescheduled and evaluated in time proportional to its length, \(O(k+1)\). Thus the total effort is \(O(mn^3)\) in the worst case; for a fixed number of cranes, this is \(O(n^3)\). The heuristic uses route construction rather than exhaustive enumeration of all possible schedules, and it does not provide an optimality guarantee.

The example below uses the specified five jobs and two cranes. The subsequent cells implement the data structures, cost evaluation, repeated insertion, schedule analysis, visualization, and feasibility checks in that order.

In [ ]:
from dataclasses import dataclass, replace
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Keep the example reproducible if random operations are added to its data setup.
np.random.seed(42)

In [ ]:
@dataclass
class Job:
    """Container-handling job with location, timing, and priority data."""

    id: int
    location: int
    processing_time: float
    release_time: float
    due_date: float
    priority_weight: float
    job_type: str


@dataclass
class Crane:
    """Yard crane with an initial bay position and equipment type."""

    id: int
    start_position: int
    crane_type: str


@dataclass
class RouteNode:
    """Start, job, or end node in an ordered crane route."""

    type: str
    position: int
    job_id: Optional[int] = None
    arrival_time: Optional[float] = None
    start_time: Optional[float] = None
    completion_time: Optional[float] = None
    insertion_order: Optional[int] = None
    insertion_cost: Optional[float] = None


@dataclass
class InsertionResult:
    """Best candidate found in one cheapest-insertion iteration."""

    job: Job
    crane_id: int
    position: int
    cost_increase: float
    new_route: List[RouteNode]
    insertion_order: Optional[int] = None

In [ ]:
def create_heuristic_example() -> Tuple[List[Job], List[Crane]]:
    """Create the five-job, two-crane example used throughout the notebook."""

    jobs = [
        Job(
            id=1,
            location=2,
            processing_time=4,
            release_time=0,
            due_date=15,
            priority_weight=2.0,
            job_type="storage",
        ),
        Job(
            id=2,
            location=5,
            processing_time=3,
            release_time=2,
            due_date=12,
            priority_weight=1.5,
            job_type="retrieval",
        ),
        Job(
            id=3,
            location=1,
            processing_time=5,
            release_time=1,
            due_date=18,
            priority_weight=1.0,
            job_type="relocation",
        ),
        Job(
            id=4,
            location=8,
            processing_time=2,
            release_time=3,
            due_date=10,
            priority_weight=2.5,
            job_type="retrieval",
        ),
        Job(
            id=5,
            location=4,
            processing_time=3,
            release_time=0,
            due_date=20,
            priority_weight=1.0,
            job_type="storage",
        ),
    ]

    cranes = [
        Crane(id=1, start_position=0, crane_type="RMG"),
        Crane(id=2, start_position=10, crane_type="RMG"),
    ]

    return jobs, cranes


jobs, cranes = create_heuristic_example()

print("=== HEURISTIC EXAMPLE SETUP ===")
print("\nJobs:")
for job in jobs:
    print(
        f"  Job {job.id}: {job.job_type.title()} at bay {job.location}, "
        f"processing time {job.processing_time} min, "
        f"release time {job.release_time} min, due date {job.due_date} min, "
        f"priority weight {job.priority_weight}"
    )

print("\nCranes:")
for crane in cranes:
    print(
        f"  Crane {crane.id}: {crane.crane_type}, "
        f"starting at bay {crane.start_position}"
    )

In [ ]:
def calculate_travel_time(pos1: int, pos2: int) -> float:
    """Return travel time in minutes at one minute per bay."""

    return float(abs(pos1 - pos2))


def initialize_crane_routes(
    cranes: List[Crane],
) -> Dict[int, List[RouteNode]]:
    """Create an empty start-to-end route for every crane."""

    routes = {}
    for crane in cranes:
        routes[crane.id] = [
            RouteNode(type="start", position=crane.start_position),
            RouteNode(type="end", position=crane.start_position),
        ]
    return routes


def schedule_route(
    route: List[RouteNode], jobs: List[Job]
) -> List[RouteNode]:
    """Return a copied route with arrival, start, and completion times set."""

    jobs_by_id = {job.id: job for job in jobs}
    scheduled_route = [replace(node) for node in route]
    start_node = scheduled_route[0]
    start_node.arrival_time = 0.0
    start_node.start_time = 0.0
    start_node.completion_time = 0.0

    for index in range(1, len(scheduled_route)):
        previous_node = scheduled_route[index - 1]
        current_node = scheduled_route[index]
        arrival_time = previous_node.completion_time + calculate_travel_time(
            previous_node.position, current_node.position
        )
        current_node.arrival_time = arrival_time

        if current_node.type == "job":
            job = jobs_by_id[current_node.job_id]
            current_node.start_time = max(arrival_time, job.release_time)
            current_node.completion_time = (
                current_node.start_time + job.processing_time
            )
        else:
            current_node.start_time = arrival_time
            current_node.completion_time = arrival_time

    return scheduled_route


def calculate_route_objective(
    route: List[RouteNode], jobs: List[Job]
) -> float:
    """Calculate route travel time plus priority-weighted job tardiness."""

    jobs_by_id = {job.id: job for job in jobs}
    travel_time = sum(
        calculate_travel_time(route[index - 1].position, route[index].position)
        for index in range(1, len(route))
    )
    weighted_tardiness = 0.0

    for node in route:
        if node.type == "job":
            job = jobs_by_id[node.job_id]
            tardiness = max(0.0, node.completion_time - job.due_date)
            weighted_tardiness += job.priority_weight * tardiness

    return travel_time + weighted_tardiness


def compute_insertion_cost(
    job: Job,
    crane_id: int,
    position: int,
    route: List[RouteNode],
    jobs: List[Job],
) -> Tuple[float, List[RouteNode]]:
    """Evaluate the objective change and scheduled route for one insertion."""

    if not 1 <= position < len(route):
        raise IndexError("Insertion position must be before the route end node.")

    original_route = schedule_route(route, jobs)
    candidate_route = [replace(node) for node in route]
    candidate_route.insert(
        position,
        RouteNode(type="job", position=job.location, job_id=job.id),
    )
    candidate_route = schedule_route(candidate_route, jobs)

    cost_increase = calculate_route_objective(
        candidate_route, jobs
    ) - calculate_route_objective(original_route, jobs)
    return cost_increase, candidate_route

In [ ]:
def find_cheapest_insertion(
    unscheduled_jobs: List[Job],
    cranes: List[Crane],
    routes: Dict[int, List[RouteNode]],
    jobs: List[Job],
) -> Optional[InsertionResult]:
    """Find the lowest-cost job, crane, and route-position combination."""

    min_cost = float("inf")
    best_insertion = None

    print("\n=== FINDING CHEAPEST INSERTION ===")

    for job in unscheduled_jobs:
        print(f"\nEvaluating Job {job.id} ({job.job_type}):")

        for crane in cranes:
            route = routes[crane.id]
            print(
                f"  Crane {crane.id}: {len(route) - 1} possible insertions"
            )

            for position in range(1, len(route)):
                cost_increase, candidate_route = compute_insertion_cost(
                    job, crane.id, position, route, jobs
                )
                print(
                    f"    Position {position}: "
                    f"objective increase = {cost_increase:.2f}"
                )

                if cost_increase < min_cost:
                    min_cost = cost_increase
                    best_insertion = InsertionResult(
                        job=job,
                        crane_id=crane.id,
                        position=position,
                        cost_increase=cost_increase,
                        new_route=candidate_route,
                    )

    if best_insertion is not None:
        print(
            f"\n*** BEST INSERTION: Job {best_insertion.job.id} on "
            f"Crane {best_insertion.crane_id} at position "
            f"{best_insertion.position}, objective increase "
            f"{best_insertion.cost_increase:.2f} ***"
        )

    return best_insertion


def insert_job_into_route(
    insertion: InsertionResult,
    routes: Dict[int, List[RouteNode]],
) -> Dict[int, List[RouteNode]]:
    """Store the evaluated route and record the insertion for inspection."""

    new_route = [replace(node) for node in insertion.new_route]
    inserted_node = next(
        node
        for node in new_route
        if node.type == "job" and node.job_id == insertion.job.id
    )
    inserted_node.insertion_order = insertion.insertion_order
    inserted_node.insertion_cost = insertion.cost_increase
    routes[insertion.crane_id] = new_route
    return routes

In [ ]:
def cheapest_insertion_heuristic(
    jobs: List[Job], cranes: List[Crane]
) -> Dict[int, List[RouteNode]]:
    """Build crane routes by repeatedly applying the cheapest insertion."""

    if not cranes and jobs:
        raise ValueError("At least one crane is required to schedule jobs.")

    print("=== CHEAPEST INSERTION HEURISTIC ===")
    print(f"Scheduling {len(jobs)} jobs with {len(cranes)} cranes.")

    routes = initialize_crane_routes(cranes)
    unscheduled_jobs = jobs.copy()
    iteration = 0

    while unscheduled_jobs:
        iteration += 1
        print(f"\n{'=' * 60}")
        print(
            f"ITERATION {iteration}: "
            f"{len(unscheduled_jobs)} jobs remaining"
        )
        print(f"{'=' * 60}")

        best_insertion = find_cheapest_insertion(
            unscheduled_jobs, cranes, routes, jobs
        )
        if best_insertion is None:
            raise RuntimeError("No insertion was available for an unscheduled job.")

        best_insertion.insertion_order = iteration
        routes = insert_job_into_route(best_insertion, routes)
        unscheduled_jobs.remove(best_insertion.job)

        print(
            f"\nJob {best_insertion.job.id} inserted into Crane "
            f"{best_insertion.crane_id} at position "
            f"{best_insertion.position}."
        )
        print(f"  Objective increase: {best_insertion.cost_increase:.2f}")

    return routes


final_routes = cheapest_insertion_heuristic(jobs, cranes)

In [ ]:
def analyze_solution(
    routes: Dict[int, List[RouteNode]], jobs: List[Job]
) -> Dict:
    """Report route schedules, tardiness, makespan, and crane utilization."""

    jobs_by_id = {job.id: job for job in jobs}
    analysis = {
        "crane_routes": {},
        "job_schedule": {},
        "performance_metrics": {},
    }

    print("\n=== SOLUTION ANALYSIS ===")
    for crane_id, route in routes.items():
        job_sequence = []
        print(f"\nCrane {crane_id} route:")

        for step, node in enumerate(route):
            if node.type == "start":
                print(f"  Step {step}: START at bay {node.position}")
            elif node.type == "end":
                print(f"  Step {step}: END at bay {node.position}")
                print(f"    Return arrival: {node.arrival_time:.1f} min")
            else:
                job = jobs_by_id[node.job_id]
                print(f"  Step {step}: Job {job.id} ({job.job_type})")
                print(f"    Bay: {node.position}")
                print(
                    f"    Arrival: {node.arrival_time:.1f} min, "
                    f"start: {node.start_time:.1f} min, "
                    f"completion: {node.completion_time:.1f} min"
                )
                print(f"    Processing time: {job.processing_time} min")

                job_sequence.append(job.id)
                analysis["job_schedule"][job.id] = {
                    "crane_id": crane_id,
                    "arrival_time": node.arrival_time,
                    "start_time": node.start_time,
                    "completion_time": node.completion_time,
                    "location": node.position,
                }

        analysis["crane_routes"][crane_id] = job_sequence

    completion_times = [
        info["completion_time"]
        for info in analysis["job_schedule"].values()
    ]
    makespan = max(completion_times, default=0.0)
    total_weighted_tardiness = 0.0

    for job_id, info in analysis["job_schedule"].items():
        job = jobs_by_id[job_id]
        tardiness = max(0.0, info["completion_time"] - job.due_date)
        weighted_tardiness = tardiness * job.priority_weight
        total_weighted_tardiness += weighted_tardiness
        print(f"\nJob {job_id} performance:")
        print(
            f"  Completion: {info['completion_time']:.1f} min; "
            f"due date: {job.due_date} min"
        )
        print(
            f"  Tardiness: {tardiness:.1f} min; "
            f"weighted tardiness: {weighted_tardiness:.2f}"
        )

    crane_utilization = {}
    for crane_id, route in routes.items():
        processing_time = sum(
            jobs_by_id[node.job_id].processing_time
            for node in route
            if node.type == "job"
        )
        crane_utilization[crane_id] = (
            100.0 * processing_time / makespan if makespan > 0 else 0.0
        )

    analysis["performance_metrics"] = {
        "makespan": makespan,
        "total_weighted_tardiness": total_weighted_tardiness,
        "crane_utilization": crane_utilization,
        "total_jobs": len(jobs),
    }

    print("\n=== PERFORMANCE SUMMARY ===")
    print(f"Makespan: {makespan:.1f} min")
    print(f"Total weighted tardiness: {total_weighted_tardiness:.2f}")
    print("Processing-time utilization over the makespan:")
    for crane_id, utilization in crane_utilization.items():
        print(f"  Crane {crane_id}: {utilization:.1f}%")

    return analysis


solution_analysis = analyze_solution(final_routes, jobs)

In [ ]:
def visualize_heuristic_solution(
    routes: Dict[int, List[RouteNode]], analysis: Dict, jobs: List[Job]
) -> None:
    """Plot the schedule, actual insertion costs, movement, and metrics."""

    jobs_by_id = {job.id: job for job in jobs}
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        "Yard Crane Scheduling: Cheapest Insertion Heuristic",
        fontsize=16,
        fontweight="bold",
    )

    # Processing bars show job work intervals; travel is shown in the movement panel.
    gantt_ax = axes[0, 0]
    colors = plt.get_cmap("tab10")
    for crane_id, route in routes.items():
        for node in route:
            if node.type == "job":
                job = jobs_by_id[node.job_id]
                gantt_ax.barh(
                    crane_id,
                    job.processing_time,
                    left=node.start_time,
                    height=0.6,
                    color=colors((job.id - 1) % 10),
                    alpha=0.85,
                )
                gantt_ax.text(
                    node.start_time + job.processing_time / 2,
                    crane_id,
                    f"J{job.id}",
                    ha="center",
                    va="center",
                    color="white",
                    fontweight="bold",
                )
    gantt_ax.set_xlabel("Time (minutes)")
    gantt_ax.set_ylabel("Crane ID")
    gantt_ax.set_title("Job Processing Schedule")
    gantt_ax.grid(True, alpha=0.3)
    gantt_ax.set_xlim(
        0, max(analysis["performance_metrics"]["makespan"] + 5, 20)
    )

    # These values come from the objective changes selected by the heuristic.
    insertion_ax = axes[0, 1]
    inserted_nodes = sorted(
        (
            node
            for route in routes.values()
            for node in route
            if node.type == "job"
        ),
        key=lambda node: node.insertion_order,
    )
    insertion_orders = [node.insertion_order for node in inserted_nodes]
    insertion_costs = [node.insertion_cost for node in inserted_nodes]
    insertion_ax.bar(insertion_orders, insertion_costs, color="steelblue", alpha=0.85)
    insertion_ax.set_xlabel("Insertion iteration")
    insertion_ax.set_ylabel("Objective increase")
    insertion_ax.set_title("Cost of the Selected Insertion")
    insertion_ax.grid(True, alpha=0.3, axis="y")
    for order, cost in zip(insertion_orders, insertion_costs):
        insertion_ax.text(
            order,
            cost,
            f"J{inserted_nodes[order - 1].job_id}: {cost:.2f}",
            ha="center",
            va="bottom" if cost >= 0 else "top",
            fontsize=8,
        )

    movement_ax = axes[1, 0]
    for crane_id, route in routes.items():
        times = [node.arrival_time for node in route]
        positions = [node.position for node in route]
        movement_ax.plot(
            times,
            positions,
            "o-",
            linewidth=2,
            markersize=6,
            label=f"Crane {crane_id}",
        )
        for node in route:
            if node.type == "job":
                movement_ax.annotate(
                    f"J{node.job_id}",
                    (node.arrival_time, node.position),
                    xytext=(4, 4),
                    textcoords="offset points",
                    fontsize=8,
                )
    movement_ax.set_xlabel("Arrival time (minutes)")
    movement_ax.set_ylabel("Bay position")
    movement_ax.set_title("Route Locations at Arrival")
    movement_ax.grid(True, alpha=0.3)
    movement_ax.legend()

    metrics_ax = axes[1, 1]
    metrics = analysis["performance_metrics"]
    metric_names = ["Makespan", "Weighted tardiness"]
    metric_values = [
        metrics["makespan"],
        metrics["total_weighted_tardiness"],
    ]
    bars = metrics_ax.bar(
        metric_names,
        metric_values,
        color=["#1f77b4", "#ff7f0e"],
        alpha=0.85,
    )
    metrics_ax.set_ylabel("Value (each metric's stated units)")
    metrics_ax.set_title("Schedule Performance Measures")
    metrics_ax.grid(True, alpha=0.3, axis="y")
    for bar, value in zip(bars, metric_values):
        metrics_ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.2f}",
            ha="center",
            va="bottom",
        )

    utilization_text = "Processing-time utilization:\n"
    for crane_id, utilization in metrics["crane_utilization"].items():
        utilization_text += f"Crane {crane_id}: {utilization:.1f}%\n"
    metrics_ax.text(
        0.02,
        0.98,
        utilization_text,
        transform=metrics_ax.transAxes,
        va="top",
        bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.8},
    )

    plt.tight_layout()
    plt.show()


visualize_heuristic_solution(final_routes, solution_analysis, jobs)

In [ ]:
def validate_solution(
    routes: Dict[int, List[RouteNode]],
    jobs: List[Job],
    cranes: List[Crane],
    analysis: Dict,
) -> None:
    """Check assignment, route timing, and reported objective components."""

    jobs_by_id = {job.id: job for job in jobs}
    crane_by_id = {crane.id: crane for crane in cranes}
    scheduled_ids = [
        node.job_id
        for route in routes.values()
        for node in route
        if node.type == "job"
    ]
    assert len(scheduled_ids) == len(jobs), "Every job must appear exactly once."
    assert set(scheduled_ids) == set(jobs_by_id), "Scheduled jobs must match the input."

    for crane_id, route in routes.items():
        assert route[0].type == "start" and route[-1].type == "end"
        assert route[0].position == crane_by_id[crane_id].start_position
        assert route[-1].position == crane_by_id[crane_id].start_position

        for index in range(1, len(route)):
            previous_node = route[index - 1]
            node = route[index]
            expected_arrival = previous_node.completion_time + calculate_travel_time(
                previous_node.position, node.position
            )
            assert np.isclose(node.arrival_time, expected_arrival)

            if node.type == "job":
                job = jobs_by_id[node.job_id]
                assert node.start_time >= job.release_time
                assert node.start_time >= node.arrival_time
                assert np.isclose(
                    node.completion_time,
                    node.start_time + job.processing_time,
                )

    expected_tardiness = sum(
        jobs_by_id[job_id].priority_weight
        * max(
            0.0,
            info["completion_time"] - jobs_by_id[job_id].due_date,
        )
        for job_id, info in analysis["job_schedule"].items()
    )
    assert np.isclose(
        expected_tardiness,
        analysis["performance_metrics"]["total_weighted_tardiness"],
    )

    route_objective = sum(
        calculate_route_objective(route, jobs) for route in routes.values()
    )
    route_travel = sum(
        calculate_travel_time(route[index - 1].position, route[index].position)
        for route in routes.values()
        for index in range(1, len(route))
    )
    assert np.isclose(route_objective, route_travel + expected_tardiness)

    print("All schedule feasibility and metric consistency checks passed.")
    print(f"Total route objective: {route_objective:.2f}")


validate_solution(final_routes, jobs, cranes, solution_analysis)

## Interpreting the heuristic

The schedule is constructed one job at a time. At every iteration, the displayed candidate costs compare the complete route objective before and after each possible insertion. A cost increase can therefore reflect added travel, tardiness changes to the inserted job, or delays and tardiness changes to jobs already in that route. The selected insertion is the least-cost candidate among those examined at that iteration.

The reported **makespan** is the latest job completion time. **Total weighted tardiness** sums each job's tardiness multiplied by its priority weight. **Processing-time utilization** is each crane's total job processing time divided by the overall makespan; it does not count travel or waiting as productive processing. These measures describe different aspects of the schedule and should be interpreted with their units in mind.

Cheapest insertion is a greedy construction heuristic, not an exact optimization method. An early low-cost insertion can restrict later choices, so the final schedule is not guaranteed to minimize the objective over all possible assignments and route orders. Its computational effort grows cubically with the number of jobs when the number of cranes is fixed. The model is intentionally limited to independent crane routes on a one-dimensional bay line; operational use may require additional constraints for crane interactions, safety, working hours, and other terminal-specific rules.